# Generate Tiny Fake M3D Dataset

This notebook generates a minimal synthetic dataset that follows the default file structure in this repo.

In [ ]:
from pathlib import Path
import json
import shutil
import numpy as np
import pandas as pd

from LaMed.src.dataset.dataset_info import dataset_info
from LaMed.src.dataset.term_dictionary import term_dict

ROOT = Path("Data/data")
CAP_DIR = ROOT / "M3D_Cap_npy"
VQA_DIR = ROOT / "M3D-VQA"
SEG_DIR = ROOT / "M3D_Seg_npy"
REFSEG_DIR = ROOT / "M3D_RefSeg_npy"
SHAPE = (1, 32, 256, 256)

for p in [CAP_DIR, VQA_DIR, SEG_DIR, REFSEG_DIR]:
    if p.exists():
        shutil.rmtree(p)

for p in [
    CAP_DIR / "images",
    CAP_DIR / "texts",
    VQA_DIR,
    SEG_DIR / "shared" / "images",
    SEG_DIR / "shared" / "labels",
    REFSEG_DIR / "masks",
]:
    p.mkdir(parents=True, exist_ok=True)


def make_image(seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    vol = rng.normal(loc=0.45, scale=0.12, size=SHAPE).astype(np.float32)
    z = np.arange(SHAPE[1], dtype=np.float32)[:, None, None]
    y = np.arange(SHAPE[2], dtype=np.float32)[None, :, None]
    x = np.arange(SHAPE[3], dtype=np.float32)[None, None, :]
    vol[0] += (0.20 * z / 31.0 + 0.12 * y / 255.0 + 0.08 * x / 255.0)

    cz, cy, cx = 14, 120 + seed % 20, 120 + (seed * 3) % 20
    rz, ry, rx = 5, 28, 28
    zz = np.arange(SHAPE[1])[:, None, None]
    yy = np.arange(SHAPE[2])[None, :, None]
    xx = np.arange(SHAPE[3])[None, None, :]
    blob = (((zz - cz) / rz) ** 2 + ((yy - cy) / ry) ** 2 + ((xx - cx) / rx) ** 2) <= 1.0
    vol[0][blob] += 0.25
    return np.clip(vol, 0.0, 1.0).astype(np.float16)


def make_mask(seed: int) -> np.ndarray:
    mask = np.zeros(SHAPE, dtype=np.uint8)
    cz, cy, cx = 14, 128 + seed % 12, 128 + (seed * 5) % 12
    rz, ry, rx = 5, 24, 24
    zz = np.arange(SHAPE[1])[:, None, None]
    yy = np.arange(SHAPE[2])[None, :, None]
    xx = np.arange(SHAPE[3])[None, None, :]
    obj = (((zz - cz) / rz) ** 2 + ((yy - cy) / ry) ** 2 + ((xx - cx) / rx) ** 2) <= 1.0
    mask[0][obj] = 1
    return mask

# Caption
cap_images = [
    ("cap_case_000.npy", make_image(100)),
    ("cap_case_001.npy", make_image(101)),
]
for name, arr in cap_images:
    np.save(CAP_DIR / "images" / name, arr)

texts = [
    "Synthetic report A: mild organ enhancement pattern and no acute finding.",
    "Synthetic report B: homogeneous soft-tissue appearance in this fake scan.",
    "Synthetic report C: no focal lesion is identified in this synthetic sample.",
    "Synthetic report D: fake volume for smoke testing loaders only.",
]
for i, txt in enumerate(texts):
    (CAP_DIR / "texts" / f"cap_case_{i:03d}.txt").write_text(txt, encoding="utf-8")

cap_entries = [
    {"image": "M3D_Cap_npy/images/cap_case_000.npy", "text": "M3D_Cap_npy/texts/cap_case_000.txt"},
    {"image": "M3D_Cap_npy/images/cap_case_001.npy", "text": "M3D_Cap_npy/texts/cap_case_001.txt"},
    {"image": "M3D_Cap_npy/images/cap_case_000.npy", "text": "M3D_Cap_npy/texts/cap_case_002.txt"},
    {"image": "M3D_Cap_npy/images/cap_case_001.npy", "text": "M3D_Cap_npy/texts/cap_case_003.txt"},
]
cap_json = {
    "train": cap_entries[:2],
    "validation": cap_entries[2:3],
    "test": cap_entries[3:],
    "hard_test": cap_entries,
    "test1k": cap_entries,
    "test500": cap_entries,
    "test100": cap_entries,
}
(CAP_DIR / "M3D_Cap.json").write_text(json.dumps(cap_json, indent=2), encoding="utf-8")
(CAP_DIR / "M3D_Cap_eh.json").write_text(json.dumps(cap_json, indent=2), encoding="utf-8")

# VQA
vqa_rows = [
    {
        "Image Path": "M3D_Cap_npy/images/cap_case_000.npy",
        "Question": "Which organ is highlighted most clearly?",
        "Choice A": "liver",
        "Choice B": "lung",
        "Choice C": "brain",
        "Choice D": "heart",
        "Answer Choice": "A",
        "Answer": "liver",
        "Question Type": 0,
    },
    {
        "Image Path": "M3D_Cap_npy/images/cap_case_001.npy",
        "Question": "Is there a focal lesion visible?",
        "Choice A": "yes",
        "Choice B": "no",
        "Choice C": "uncertain",
        "Choice D": "not applicable",
        "Answer Choice": "B",
        "Answer": "no",
        "Question Type": 1,
    },
    {
        "Image Path": "M3D_Cap_npy/images/cap_case_000.npy",
        "Question": "Describe the overall finding briefly.",
        "Choice A": "normal",
        "Choice B": "abnormal",
        "Choice C": "artifact",
        "Choice D": "unknown",
        "Answer Choice": "A",
        "Answer": "normal appearance with mild intensity variation",
        "Question Type": 2,
    },
    {
        "Image Path": "M3D_Cap_npy/images/cap_case_001.npy",
        "Question": "What modality does this synthetic data mimic?",
        "Choice A": "CT",
        "Choice B": "MRI",
        "Choice C": "PET",
        "Choice D": "ultrasound",
        "Answer Choice": "A",
        "Answer": "CT-like synthetic volume",
        "Question Type": 3,
    },
]
pd.DataFrame(vqa_rows[:2]).to_csv(VQA_DIR / "M3D_VQA_train.csv", index=False)
pd.DataFrame(vqa_rows[2:3]).to_csv(VQA_DIR / "M3D_VQA_val.csv", index=False)
pd.DataFrame(vqa_rows[3:]).to_csv(VQA_DIR / "M3D_VQA_test.csv", index=False)

vqa_yn_rows = [
    {
        "Image Path": "M3D_Cap_npy/images/cap_case_000.npy",
        "Question": "Is the liver region present in this volume?",
        "Answer": "yes",
        "Answer Choice": "A",
        "Question Type": 10,
    },
    {
        "Image Path": "M3D_Cap_npy/images/cap_case_001.npy",
        "Question": "Is there severe motion artifact?",
        "Answer": "no",
        "Answer Choice": "B",
        "Question Type": 11,
    },
]
pd.DataFrame(vqa_yn_rows).to_csv(VQA_DIR / "M3D_VQA_yn_train.csv", index=False)

# Segmentation shared files
shared_image_rel = "shared/images/shared_case.npy"
shared_label_rel = "shared/labels/shared_0.npy"
np.save(SEG_DIR / shared_image_rel, make_image(200))
np.save(SEG_DIR / shared_label_rel, make_mask(500))

dataset_info_json = {k: v for k, v in sorted(dataset_info.items(), key=lambda kv: kv[0])}
for tag in dataset_info_json:
    tag_dir = SEG_DIR / tag
    tag_dir.mkdir(parents=True, exist_ok=True)
    datalist = [{"image": shared_image_rel, "label": shared_label_rel}]
    doc = {
        "name": f"fake_m3d_{tag}",
        "description": "Tiny fake segmentation set for smoke tests.",
        "labels": {"0": "background", "1": dataset_info_json[tag][0]},
        "train": datalist,
        "training": datalist,
        "test": datalist,
    }
    (tag_dir / f"{tag}.json").write_text(json.dumps(doc, indent=2), encoding="utf-8")

(SEG_DIR / "dataset_info.json").write_text(json.dumps(dataset_info_json, indent=2), encoding="utf-8")
(SEG_DIR / "term_dictionary.json").write_text(json.dumps(term_dict, indent=2), encoding="utf-8")

# RefSeg
ref_mask_rel = "M3D_RefSeg_npy/masks/ref_case_000.npy"
np.save(ROOT / ref_mask_rel, make_mask(700))
ref_rows = [
    {
        "Image": "M3D_Cap_npy/images/cap_case_000.npy",
        "Mask": ref_mask_rel,
        "Mask_ID": 1,
        "Question": "Segment the highlighted liver region.",
        "Answer": "The liver region is segmented as requested.",
    }
]
pd.DataFrame(ref_rows).to_csv(REFSEG_DIR / "M3D_RefSeg.csv", index=False)
pd.DataFrame(ref_rows).to_csv(REFSEG_DIR / "M3D_RefSeg_test.csv", index=False)

print("Fake dataset generation complete.")